# 04 · Robustness evaluation (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1ow1atency/floodfusion/blob/main/notebooks/04_evaluate.ipynb)

**Final reporting only: nothing here may be used to tune hyperparameters or to select models.** Every model is its `best.pt`, selected earlier on the real valid split.

Evaluates every run folder under `floodfusion/runs` on test and bolivia: synthetic cloud coverage 0-100% (3 cloud seeds, identical clouds for every model) in two variants, S2 or S1 removed, and real-cloud bins. **Thick** clouds are the ones used in robustness training; they are semi-transparent (median opacity ~0.8, water stays visible through them, see `docs/cloud_leakage.md`). **Opaque** clouds use the same masks with opacity 1 inside, so no ground signal survives: the stricter test. It first checks that coverage 0 reproduces each run's `eval_test` / `eval_bolivia` exactly, so run notebook 03 to the end for every run first. Results go to `floodfusion/results/` on Drive. Needs a **T4 GPU** runtime.

**Expected runtime (T4, 7 runs): about 15-25 min** for the first run (a few minutes of setup and data copy, then ~10-15 min of evaluation: 35 settings per split). Re-runs in the same session are a little faster, because the clouds come from `CACHE_DIR`. The evaluation cell prints the actual time.

Settings.

In [ ]:
RUNS_DIR = '/content/drive/MyDrive/floodfusion/runs'        # every <RUNS_DIR>/*/best.pt is evaluated
RESULTS_DIR = '/content/drive/MyDrive/floodfusion/results'  # robustness.csv, summary.md, figures
CACHE_DIR = '/content/cloud_cache'                          # generated clouds on local disk (~4 GB), reused on re-runs

GPU check.

In [ ]:
!nvidia-smi
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU available. In Colab: Runtime → Change runtime type → T4 GPU, then run all cells again.')
print('GPU:', torch.cuda.get_device_name(0))

Mount Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Clone the repo, or pull if it is already cloned. Uses the `GITHUB_TOKEN` Colab secret if set (see README "Colab setup"); otherwise tries a plain public clone.

In [ ]:
# Clone or update the repo in Colab. Shared cell: paste verbatim into every notebook
# (tests/test_notebooks.py checks the copies match notebooks/colab_clone_cell.py).
# Uses the Colab secret GITHUB_TOKEN if available (see README "Colab setup"). The token
# reaches git only through environment variables: never on the command line, never in
# .git/config (the remote URL stays token-free), and it is masked in any output.
import base64, os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/1ow1atency/floodfusion.git'
REPO_DIR = Path('/content/floodfusion')


def _github_token():
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception as e:  # secret missing, notebook access off, or not in Colab
        print(f'GITHUB_TOKEN secret not available ({type(e).__name__}); trying a plain public clone.\n'
              'To add it: key icon in the left sidebar -> "Add new secret" -> name GITHUB_TOKEN, value = a\n'
              'GitHub token with read access to this repo -> switch "Notebook access" on -> re-run this cell.')
        return None
    return token.strip() or None


def _git(*args, token=None):
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}  # fail instead of waiting for a password
    secrets = []
    if token:
        b64 = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0=f'Authorization: Basic {b64}')
        secrets = [token, b64]
    r = subprocess.run(['git', *args], env=env, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    for s in secrets:
        out = out.replace(s, '***')
    if r.returncode != 0:
        hint = '' if token else '\n(No token used. If the repo is not publicly visible, add the GITHUB_TOKEN secret.)'
        raise RuntimeError(f'git {args[0] if args[0] != "-C" else args[2]} failed:\n{out}{hint}')
    return out


def _clone_or_pull():
    version = tuple(int(x) for x in _git('--version').split()[2].split('.')[:2])
    if version < (2, 31):  # GIT_CONFIG_COUNT needs git >= 2.31
        raise RuntimeError(f'git >= 2.31 needed, found {version}')
    token = _github_token()
    if (REPO_DIR / '.git').exists():
        _git('-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL)  # keep the remote token-free
        print(_git('-C', str(REPO_DIR), 'pull', '--ff-only', token=token))
    else:
        print(_git('clone', REPO_URL, str(REPO_DIR), token=token))
    print('at', _git('-C', str(REPO_DIR), 'log', '--oneline', '-1'))


_clone_or_pull()

Install the package (base dependencies only).

In [ ]:
!pip install -q -e "{REPO_DIR}"
assert _exit_code == 0, 'pip install failed, see output above'

Copy the dataset from Drive to the local disk (much faster to read every epoch). Skipped if a complete copy is already there.

In [ ]:
import shutil, time

DRIVE_DATA = Path('/content/drive/MyDrive/floodfusion/data/sen1floods11')
LOCAL_DATA = Path('/content/data/sen1floods11')
FOLDERS, EXPECTED = ['S1Hand', 'S2Hand', 'LabelHand'], 446


def tif_counts(root):
    return {f: len(list((root / f).glob('*.tif'))) if (root / f).is_dir() else 0 for f in FOLDERS}


marker = LOCAL_DATA / '.copy_complete'
if marker.exists() and all(n == EXPECTED for n in tif_counts(LOCAL_DATA).values()):
    print(f'local copy already present: {LOCAL_DATA}')
else:
    t0 = time.time()
    for f in FOLDERS + ['splits']:
        shutil.copytree(DRIVE_DATA / f, LOCAL_DATA / f, dirs_exist_ok=True)
    print(f'copied {DRIVE_DATA} -> {LOCAL_DATA} in {time.time() - t0:.0f} s')
counts = tif_counts(LOCAL_DATA)
csvs = sorted(p.name for p in (LOCAL_DATA / 'splits').glob('*.csv'))
print('tif files per folder:', counts, '| split CSVs:', csvs)
assert all(n == EXPECTED for n in counts.values()), f'expected {EXPECTED} .tif files per folder, got {counts}'
assert len(csvs) == 4, f'expected 4 split CSVs, got {csvs}'
marker.touch()

Run the evaluation. Progress is printed per setting; the sanity check stops the run if coverage 0 does not reproduce a run's existing evaluation.

In [ ]:
t0 = time.time()
!cd {REPO_DIR} && python scripts/robustness_eval.py --runs "{RUNS_DIR}" --root "{LOCAL_DATA}" --out "{RESULTS_DIR}" --cache-dir "{CACHE_DIR}"
assert _exit_code == 0, 'robustness_eval.py failed, see output above'
print(f'done in {(time.time() - t0) / 60:.1f} min')

Summary table and figures (also saved in the results folder).

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((Path(RESULTS_DIR) / 'summary.md').read_text()))
for name in ['iou_vs_coverage_test.png', 'iou_vs_coverage_bolivia.png', 'real_cloud_bins.png', 'qualitative.png']:
    path = Path(RESULTS_DIR) / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))